## Section 1: Setup

In [1]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    import duckdb
except ImportError:
    pip_install("duckdb")
    import duckdb

try:
    import pandas as pd
except ImportError:
    pip_install("pandas")
    import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    pip_install("matplotlib")
    import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    pip_install("seaborn")
    import seaborn as sns

import os
import requests
import zipfile
import io

con = duckdb.connect()
print("DuckDB connected:", duckdb.__version__)

DuckDB connected: 1.5.2


## Section 2: Confirm column names

In [2]:
diag_schema = con.execute("DESCRIBE SELECT * FROM 'data/diagnoses_icd.csv' LIMIT 1").df()
print(diag_schema)

required = {"subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"}
found = set(diag_schema["column_name"].tolist())
missing = required - found
if missing:
    raise RuntimeError(f"Missing required columns in diagnoses_icd.csv: {missing}")
print("\nAll required columns confirmed.")

   column_name column_type null   key default extra
0   subject_id      BIGINT  YES  None    None  None
1      hadm_id      BIGINT  YES  None    None  None
2      seq_num      BIGINT  YES  None    None  None
3     icd_code     VARCHAR  YES  None    None  None
4  icd_version      BIGINT  YES  None    None  None

All required columns confirmed.


## Section 3: Download CCSR ICD-10 Mapping

In [3]:
import os, requests, zipfile, glob as _glob

CCSR_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccsr/DXCCSR_v2025-1.zip"
CCSR_ZIP_PATH = "ccsr/DXCCSR_v2025-1.zip"
CCSR_EXTRACT_DIR = "ccsr/icd10_ccsr"

os.makedirs("ccsr", exist_ok=True)
os.makedirs(CCSR_EXTRACT_DIR, exist_ok=True)

if not os.path.exists(CCSR_ZIP_PATH):
    print("Downloading CCSR ICD-10 mapping from AHRQ...")
    resp = requests.get(CCSR_ZIP_URL, timeout=120)
    resp.raise_for_status()
    with open(CCSR_ZIP_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded: {len(resp.content) / 1024:.1f} KB")
else:
    print(f"Already exists: {CCSR_ZIP_PATH}")

if not _glob.glob(f"{CCSR_EXTRACT_DIR}/*.csv"):
    with zipfile.ZipFile(CCSR_ZIP_PATH, "r") as zf:
        zf.extractall(CCSR_EXTRACT_DIR)
    print("Extracted.")
else:
    print("Already extracted.")

print("\nFiles in ccsr/icd10_ccsr/:")
for f in os.listdir(CCSR_EXTRACT_DIR):
    print(" ", f)

Already exists: ccsr/DXCCSR_v2025-1.zip
Already extracted.

Files in ccsr/icd10_ccsr/:
  DXCCSR-ChangeLog-v20241-v20251.xlsx
  DXCCSR-Reference-File-v2025-1.xlsx
  DXCCSR-User-Guide-v2025-1.pdf
  DXCCSR_Mapping_Program_v2025-1.sas
  DXCCSR_v2025-1.csv


## Section 3b: Download CCS ICD-9 Mapping

In [4]:
CCS_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccs/Single_Level_CCS_2015.zip"
CCS_ZIP_PATH = "ccsr/ccs_icd9.zip"
CCS_EXTRACT_DIR = "ccsr/icd9_ccs"

os.makedirs(CCS_EXTRACT_DIR, exist_ok=True)

if not os.path.exists(CCS_ZIP_PATH):
    print("Downloading CCS ICD-9 mapping from AHRQ...")
    resp = requests.get(CCS_ZIP_URL, timeout=120)
    resp.raise_for_status()
    with open(CCS_ZIP_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded: {len(resp.content) / 1024:.1f} KB")
else:
    print(f"Already exists: {CCS_ZIP_PATH}")

if not _glob.glob(f"{CCS_EXTRACT_DIR}/*.csv"):
    with zipfile.ZipFile(CCS_ZIP_PATH, "r") as zf:
        zf.extractall(CCS_EXTRACT_DIR)
    print("Extracted.")
else:
    print("Already extracted.")

ccs_csvs = _glob.glob(f"{CCS_EXTRACT_DIR}/*.csv")
if not ccs_csvs:
    raise RuntimeError("No CSV found in ccsr/icd9_ccs/ after extraction.")

ccs_csv_path = ccs_csvs[0]
print(f"\nUsing: {ccs_csv_path}")

ccs_raw = pd.read_csv(ccs_csv_path, skiprows=1, nrows=3)
print("\nActual columns after skipping note row:")
print(ccs_raw.columns.tolist())
print("\nFirst 3 rows:")
print(ccs_raw)

con.register("ccs_icd9_map_raw", pd.read_csv(ccs_csv_path, skiprows=1))
con.execute("CREATE OR REPLACE TABLE ccs_icd9_map AS SELECT * FROM ccs_icd9_map_raw")
print(f"\nccs_icd9_map loaded: {con.execute('SELECT COUNT(*) FROM ccs_icd9_map').fetchone()[0]:,} rows")

Already exists: ccsr/ccs_icd9.zip
Already extracted.

Using: ccsr/icd9_ccs\$dxref 2015.csv

Actual columns after skipping note row:
["'ICD-9-CM CODE'", "'CCS CATEGORY'", "'CCS CATEGORY DESCRIPTION'", "'ICD-9-CM CODE DESCRIPTION'", "'OPTIONAL CCS CATEGORY'", "'OPTIONAL CCS CATEGORY DESCRIPTION'"]

First 3 rows:
  'ICD-9-CM CODE' 'CCS CATEGORY' 'CCS CATEGORY DESCRIPTION'  \
0         '     '        '0    '                    'No DX'   
1         '01000'        '1    '             'Tuberculosis'   
2         '01001'        '1    '             'Tuberculosis'   

  'ICD-9-CM CODE DESCRIPTION' 'OPTIONAL CCS CATEGORY'  \
0  INVALID CODES IN USER DATA                     ' '   
1      PRIM TB COMPLEX-UNSPEC                     ' '   
2     PRIM TB COMPLEX-NO EXAM                     ' '   

  'OPTIONAL CCS CATEGORY DESCRIPTION'  
0                                 ' '  
1                                 ' '  
2                                 ' '  

ccs_icd9_map loaded: 15,073 rows


## Section 4: Load CCSR ICD-10 Mapping into DuckDB

In [5]:
ccsr_csvs = _glob.glob("ccsr/icd10_ccsr/DXCCSR*.csv")
if not ccsr_csvs:
    raise RuntimeError("No DXCCSR*.csv found in ccsr/icd10_ccsr/")

ccsr_csv_path = ccsr_csvs[0]
print(f"Using: {ccsr_csv_path}")

ccsr_raw = pd.read_csv(ccsr_csv_path, nrows=3)
print("\nActual columns:")
print(ccsr_raw.columns.tolist())
print("\nFirst 3 rows:")
print(ccsr_raw)

# Column 0: ICD-10-CM CODE (join key)
# Column 1: ICD-10-CM CODE DESCRIPTION (free text, skip)
# Column 2: Default CCSR CATEGORY IP (the actual category)
icd10_col = ccsr_raw.columns[0]
ccsr_cat_col = ccsr_raw.columns[2]
print(f"\nJoin key column  : '{icd10_col}'")
print(f"Category column  : '{ccsr_cat_col}'")

ccsr_full = pd.read_csv(ccsr_csv_path)
ccsr_full[icd10_col] = ccsr_full[icd10_col].astype(str).str.strip().str.replace("'", "", regex=False)
con.register("ccsr_map_raw", ccsr_full)
con.execute("CREATE OR REPLACE TABLE ccsr_map AS SELECT * FROM ccsr_map_raw")
print(f"ccsr_map loaded: {con.execute('SELECT COUNT(*) FROM ccsr_map').fetchone()[0]:,} rows")

Using: ccsr/icd10_ccsr\DXCCSR_v2025-1.csv

Actual columns:
["'ICD-10-CM CODE'", "'ICD-10-CM CODE DESCRIPTION'", "'Default CCSR CATEGORY IP'", "'Default CCSR CATEGORY DESCRIPTION IP'", "'Default CCSR CATEGORY OP'", "'Default CCSR CATEGORY DESCRIPTION OP'", "'CCSR CATEGORY 1'", "'CCSR CATEGORY 1 DESCRIPTION'", "'CCSR CATEGORY 2'", "'CCSR CATEGORY 2 DESCRIPTION'", "'CCSR CATEGORY 3'", "'CCSR CATEGORY 3 DESCRIPTION'", "'CCSR CATEGORY 4'", "'CCSR CATEGORY 4 DESCRIPTION'", "'CCSR CATEGORY 5'", "'CCSR CATEGORY 5 DESCRIPTION'", "'CCSR CATEGORY 6'", "'CCSR CATEGORY 6 DESCRIPTION'", "'Rationale for Default Assignment'"]

First 3 rows:
  'ICD-10-CM CODE'                       'ICD-10-CM CODE DESCRIPTION'  \
0           'A000'  Cholera due to Vibrio cholerae 01, biovar chol...   
1           'A001'    Cholera due to Vibrio cholerae 01, biovar eltor   
2           'A009'                               Cholera, unspecified   

  'Default CCSR CATEGORY IP' 'Default CCSR CATEGORY DESCRIPTION IP'  \
0  

C:\Users\dhara\AppData\Local\Temp\ipykernel_36932\3620806751.py:22: DtypeWarning: Columns (15,17) have mixed types. Specify dtype option on import or set low_memory=False.
  ccsr_full = pd.read_csv(ccsr_csv_path)


## Section 5: Filter Diagnoses to Cohort

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE cohort_diagnoses AS
    SELECT d.subject_id, d.hadm_id, d.seq_num, d.icd_code, d.icd_version
    FROM 'data/diagnoses_icd.csv' d
    INNER JOIN 'outputs/cohort_10k.csv' c ON d.subject_id = c.subject_id
""")

total_rows = con.execute("SELECT COUNT(*) FROM cohort_diagnoses").fetchone()[0]
distinct_patients = con.execute("SELECT COUNT(DISTINCT subject_id) FROM cohort_diagnoses").fetchone()[0]
print(f"Total diagnosis rows for cohort : {total_rows:,}")
print(f"Distinct patients with >= 1 diag: {distinct_patients:,}")
print()
print("ICD version breakdown:")
print(con.execute("""
    SELECT icd_version, COUNT(*) AS count
    FROM cohort_diagnoses
    GROUP BY icd_version
    ORDER BY icd_version
""").df().to_string(index=False))

Total diagnosis rows for cohort : 290,683
Distinct patients with >= 1 diag: 9,992

ICD version breakdown:
 icd_version  count
           9 132413
          10 158270


## Section 6: Map ICD Codes to Clinical Categories

In [7]:
# Step 1: Map ICD-10 via CCSR
con.execute(f"""
    CREATE OR REPLACE TABLE icd10_mapped AS
    SELECT d.subject_id, d.hadm_id, d.icd_code, d.icd_version,
        m."{ccsr_cat_col}" AS ccsr_category
    FROM cohort_diagnoses d
    LEFT JOIN ccsr_map m ON d.icd_code = m."{icd10_col}"
    WHERE d.icd_version = 10
""")
icd10_ok = con.execute("SELECT COUNT(*) FROM icd10_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
icd10_no = con.execute("SELECT COUNT(*) FROM icd10_mapped WHERE ccsr_category IS NULL").fetchone()[0]
print(f"ICD-10 mapped  : {icd10_ok:,}")
print(f"ICD-10 unmapped: {icd10_no:,}")
print()

# Step 2: Map ICD-9 via CCS
ccs_cols = con.execute("DESCRIBE ccs_icd9_map").df()["column_name"].tolist()
print(f"CCS ICD-9 columns: {ccs_cols}")
ccs_icd9_code_col = ccs_cols[0]
ccs_cat_candidates = [c for c in ccs_cols if "DESCRIPTION" in c.upper() or "CATEGORY" in c.upper()]
if not ccs_cat_candidates:
    raise RuntimeError(f"Cannot find category column. Columns: {ccs_cols}")
ccs_cat_col_icd9 = ccs_cat_candidates[0]
print(f"ICD-9 code column   : '{ccs_icd9_code_col}'")
print(f"ICD-9 category column: '{ccs_cat_col_icd9}'")
print()

con.execute(f"""
    CREATE OR REPLACE TABLE icd9_mapped AS
    SELECT d.subject_id, d.hadm_id, d.icd_code, d.icd_version,
        m."{ccs_cat_col_icd9}" AS ccsr_category
    FROM cohort_diagnoses d
    LEFT JOIN (
        SELECT
            TRIM(REPLACE(REPLACE("{ccs_icd9_code_col}", chr(39), ''), '"', '')) AS icd9_clean,
            "{ccs_cat_col_icd9}"
        FROM ccs_icd9_map
    ) m ON d.icd_code = m.icd9_clean
    WHERE d.icd_version = 9
""")
icd9_ok = con.execute("SELECT COUNT(*) FROM icd9_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
icd9_no = con.execute("SELECT COUNT(*) FROM icd9_mapped WHERE ccsr_category IS NULL").fetchone()[0]
print(f"ICD-9 mapped  : {icd9_ok:,}")
print(f"ICD-9 unmapped: {icd9_no:,}")
print()

# Step 3: Combine into all_mapped
con.execute("""
    CREATE OR REPLACE TABLE all_mapped AS
    SELECT * FROM icd10_mapped
    UNION ALL
    SELECT * FROM icd9_mapped
""")
total_ok = con.execute("SELECT COUNT(*) FROM all_mapped WHERE ccsr_category IS NOT NULL").fetchone()[0]
total_no = con.execute("SELECT COUNT(*) FROM all_mapped WHERE ccsr_category IS NULL").fetchone()[0]
total = total_ok + total_no
rate = total_ok / total * 100 if total > 0 else 0
print(f"Total mapped rows  : {total_ok:,}")
print(f"Total unmapped rows: {total_no:,}")
print(f"Overall mapping rate: {rate:.1f}%")

ICD-10 mapped  : 158,270
ICD-10 unmapped: 0

CCS ICD-9 columns: ["'ICD-9-CM CODE'", "'CCS CATEGORY'", "'CCS CATEGORY DESCRIPTION'", "'ICD-9-CM CODE DESCRIPTION'", "'OPTIONAL CCS CATEGORY'", "'OPTIONAL CCS CATEGORY DESCRIPTION'"]
ICD-9 code column   : ''ICD-9-CM CODE''
ICD-9 category column: ''CCS CATEGORY''

ICD-9 mapped  : 132,413
ICD-9 unmapped: 0

Total mapped rows  : 290,683
Total unmapped rows: 0
Overall mapping rate: 100.0%


## Section 7: Build Patient-Level Feature Table

In [8]:
categories = con.execute("""
    SELECT DISTINCT ccsr_category
    FROM all_mapped
    WHERE ccsr_category IS NOT NULL
    ORDER BY ccsr_category
""").df()["ccsr_category"].tolist()
print(f"Distinct categories found: {len(categories)}")

mapped_df = con.execute("""
    SELECT subject_id, ccsr_category
    FROM all_mapped
    WHERE ccsr_category IS NOT NULL
""").df()

pivot = (
    mapped_df
    .assign(val=1)
    .drop_duplicates(subset=["subject_id", "ccsr_category"])
    .pivot(index="subject_id", columns="ccsr_category", values="val")
    .fillna(0)
    .astype(int)
    .reset_index()
)
pivot.columns.name = None

cohort_ids = con.execute("SELECT DISTINCT subject_id FROM 'outputs/cohort_10k.csv'").df()
pivot = cohort_ids.merge(pivot, on="subject_id", how="left").fillna(0)
pivot["subject_id"] = pivot["subject_id"].astype(int)
feature_cols = [c for c in pivot.columns if c != "subject_id"]
pivot[feature_cols] = pivot[feature_cols].astype(int)

print(f"Feature table shape: {pivot.shape}")
print(f"Expected: (9996, 530 to 600)")

Distinct categories found: 707
Feature table shape: (9996, 708)
Expected: (9996, 530 to 600)


## Section 8: Save as Parquet

In [10]:
# Section 8: Rename clinical category columns and save as Parquet

import os
import glob
import zipfile
import pandas as pd
from collections import Counter

try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests


# -----------------------------
# Emergency setup checks
# -----------------------------

if "CCSR_ZIP_URL" not in locals():
    CCSR_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccsr/DXCCSR_v2025-1.zip"

if "CCSR_ZIP_PATH" not in locals():
    CCSR_ZIP_PATH = "ccsr/DXCCSR_v2025-1.zip"

if "CCSR_EXTRACT_DIR" not in locals():
    CCSR_EXTRACT_DIR = "ccsr/icd10_ccsr"

if "CCS_ZIP_URL" not in locals():
    CCS_ZIP_URL = "https://hcup-us.ahrq.gov/toolssoftware/ccs/Single_Level_CCS_2015.zip"

if "CCS_ZIP_PATH" not in locals():
    CCS_ZIP_PATH = "ccsr/ccs_icd9.zip"

if "CCS_EXTRACT_DIR" not in locals():
    CCS_EXTRACT_DIR = "ccsr/icd9_ccs"


# -----------------------------
# Reload ICD-10 CCSR mapping if needed
# -----------------------------

os.makedirs("ccsr", exist_ok=True)
os.makedirs(CCSR_EXTRACT_DIR, exist_ok=True)

if "ccsr_full" not in locals():
    if not os.path.exists(CCSR_ZIP_PATH):
        print("Downloading ICD-10 CCSR mapping...")
        resp = requests.get(CCSR_ZIP_URL, timeout=120)
        resp.raise_for_status()
        with open(CCSR_ZIP_PATH, "wb") as f:
            f.write(resp.content)

    if not glob.glob(f"{CCSR_EXTRACT_DIR}/*.csv"):
        with zipfile.ZipFile(CCSR_ZIP_PATH, "r") as zf:
            zf.extractall(CCSR_EXTRACT_DIR)

    ccsr_csvs = glob.glob(f"{CCSR_EXTRACT_DIR}/DXCCSR*.csv")
    if not ccsr_csvs:
        raise RuntimeError("No DXCCSR*.csv file found for ICD-10 CCSR mapping.")

    ccsr_csv_path = ccsr_csvs[0]
    ccsr_full = pd.read_csv(ccsr_csv_path, low_memory=False)
    print(f"Reloaded ICD-10 CCSR mapping from: {ccsr_csv_path}")


# -----------------------------
# Reload ICD-9 CCS mapping if needed
# -----------------------------

os.makedirs(CCS_EXTRACT_DIR, exist_ok=True)

if "ccs_csv_path" not in locals():
    if not os.path.exists(CCS_ZIP_PATH):
        print("Downloading ICD-9 CCS mapping...")
        resp = requests.get(CCS_ZIP_URL, timeout=120)
        resp.raise_for_status()
        with open(CCS_ZIP_PATH, "wb") as f:
            f.write(resp.content)

    if not glob.glob(f"{CCS_EXTRACT_DIR}/*.csv"):
        with zipfile.ZipFile(CCS_ZIP_PATH, "r") as zf:
            zf.extractall(CCS_EXTRACT_DIR)

    ccs_csvs = glob.glob(f"{CCS_EXTRACT_DIR}/*.csv")
    if not ccs_csvs:
        raise RuntimeError("No CSV file found for ICD-9 CCS mapping.")

    ccs_csv_path = ccs_csvs[0]


# -----------------------------
# Confirm final feature dataframe
# -----------------------------

if "pivot" not in locals():
    raise RuntimeError(
        "The final feature dataframe is named 'pivot' in Section 7. "
        "Run Section 7 first, then run this Section 8 cell."
    )


# -----------------------------
# Helper functions
# -----------------------------

def clean_col_name(x):
    return str(x).replace("'", "").replace('"', "").strip().upper()

def clean_key(x):
    return str(x).strip().strip("'").strip('"').strip()

def clean_description(x):
    return str(x).strip().strip("'").strip('"').strip()


# -----------------------------
# Build ICD-10 CCSR lookup
# -----------------------------

ccsr_cols = list(ccsr_full.columns)

ccsr_cat_candidates = [
    c for c in ccsr_cols
    if "CCSR" in clean_col_name(c)
    and "CATEGORY" in clean_col_name(c)
    and "DESCRIPTION" not in clean_col_name(c)
    and "IP" in clean_col_name(c)
]

ccsr_desc_candidates = [
    c for c in ccsr_cols
    if "CCSR" in clean_col_name(c)
    and "CATEGORY" in clean_col_name(c)
    and "DESCRIPTION" in clean_col_name(c)
    and "IP" in clean_col_name(c)
]

if not ccsr_cat_candidates:
    ccsr_cat_col = ccsr_full.columns[2]
else:
    ccsr_cat_col = ccsr_cat_candidates[0]

if not ccsr_desc_candidates:
    ccsr_desc_col = ccsr_full.columns[3]
else:
    ccsr_desc_col = ccsr_desc_candidates[0]

print("ICD-10 category column:", ccsr_cat_col)
print("ICD-10 description column:", ccsr_desc_col)

ccsr10_pairs = (
    ccsr_full[[ccsr_cat_col, ccsr_desc_col]]
    .dropna(subset=[ccsr_cat_col])
    .drop_duplicates(subset=[ccsr_cat_col])
)

icd10_lookup = {}

for _, row in ccsr10_pairs.iterrows():
    raw_code = str(row[ccsr_cat_col]).strip()
    clean_code = clean_key(row[ccsr_cat_col])
    desc = clean_description(row[ccsr_desc_col])

    icd10_lookup[raw_code] = desc
    icd10_lookup[clean_code] = desc


# -----------------------------
# Build ICD-9 CCS lookup
# -----------------------------

ccs_icd9_df = pd.read_csv(ccs_csv_path, skiprows=1, low_memory=False)
ccs_cols = list(ccs_icd9_df.columns)

ccs_cat_candidates = [
    c for c in ccs_cols
    if "CCS" in clean_col_name(c)
    and "CATEGORY" in clean_col_name(c)
    and "DESCRIPTION" not in clean_col_name(c)
]

ccs_desc_candidates = [
    c for c in ccs_cols
    if "CCS" in clean_col_name(c)
    and "CATEGORY" in clean_col_name(c)
    and "DESCRIPTION" in clean_col_name(c)
]

if not ccs_cat_candidates:
    ccs_cat_col = ccs_icd9_df.columns[1]
else:
    ccs_cat_col = ccs_cat_candidates[0]

if not ccs_desc_candidates:
    ccs_desc_col = ccs_icd9_df.columns[2]
else:
    ccs_desc_col = ccs_desc_candidates[0]

print("ICD-9 category column:", ccs_cat_col)
print("ICD-9 description column:", ccs_desc_col)

icd9_pairs = (
    ccs_icd9_df[[ccs_cat_col, ccs_desc_col]]
    .dropna(subset=[ccs_cat_col])
    .drop_duplicates(subset=[ccs_cat_col])
)

icd9_lookup = {}

for _, row in icd9_pairs.iterrows():
    raw_code = str(row[ccs_cat_col]).strip()
    clean_code = clean_key(row[ccs_cat_col])
    desc = clean_description(row[ccs_desc_col])

    icd9_lookup[raw_code] = desc
    icd9_lookup[clean_code] = desc


# -----------------------------
# Combine lookups
# ICD-10 overrides ICD-9 if overlap happens
# -----------------------------

combined_lookup = {}
combined_lookup.update(icd9_lookup)
combined_lookup.update(icd10_lookup)

print(f"Combined lookup size: {len(combined_lookup)} entries")


# -----------------------------
# Rename pivot columns
# -----------------------------

feature_cols = [c for c in pivot.columns if c != "subject_id"]

print()
print("Before rename, sample 5 feature column names:")
for c in feature_cols[:5]:
    matched_name = combined_lookup.get(str(c).strip(), combined_lookup.get(clean_key(c), "(no match)"))
    print(f"  {repr(c)} -> {matched_name}")

raw_rename_map = {}

for c in feature_cols:
    direct_key = str(c).strip()
    cleaned_key = clean_key(c)

    if direct_key in combined_lookup:
        raw_rename_map[c] = combined_lookup[direct_key]
    elif cleaned_key in combined_lookup:
        raw_rename_map[c] = combined_lookup[cleaned_key]

# Protect against duplicate clinical names
name_counts = Counter(raw_rename_map.values())
rename_map = {}

for old_name, new_name in raw_rename_map.items():
    if name_counts[new_name] > 1:
        rename_map[old_name] = f"{new_name} ({clean_key(old_name)})"
    else:
        rename_map[old_name] = new_name

renamed_count = len(rename_map)
total_count = len(feature_cols)

pivot = pivot.rename(columns=rename_map)

duplicate_cols = pivot.columns[pivot.columns.duplicated()].tolist()
if duplicate_cols:
    raise RuntimeError(f"Duplicate column names found after renaming: {duplicate_cols[:10]}")

feature_cols_after = [c for c in pivot.columns if c != "subject_id"]

print()
print("After rename, sample 5 feature column names:")
for c in feature_cols_after[:5]:
    print(f"  {repr(c)}")

print()
print(f"Renamed {renamed_count} out of {total_count} diagnosis feature columns")

unmatched_cols = [c for c in feature_cols if c not in rename_map]

if unmatched_cols:
    print()
    print(f"Unmatched columns: {len(unmatched_cols)}")
    print("Sample unmatched columns:")
    for c in unmatched_cols[:10]:
        print(f"  {repr(c)}")


# -----------------------------
# Save output files
# -----------------------------

os.makedirs("outputs", exist_ok=True)

parquet_path = "outputs/ccsr_features.parquet"
preview_path = "outputs/ccsr_features_preview.csv"

pivot.to_parquet(parquet_path, index=False)
pivot.head(20).to_csv(preview_path, index=False)

print()
print(f"Saved: {parquet_path} ({os.path.getsize(parquet_path) / 1024:.1f} KB)")
print(f"Saved: {preview_path} ({os.path.getsize(preview_path) / 1024:.1f} KB)")

assert os.path.exists(parquet_path), "Parquet missing"
assert os.path.exists(preview_path), "Preview CSV missing"

print("Both files confirmed on disk.")

ICD-10 category column: 'Default CCSR CATEGORY IP'
ICD-10 description column: 'Default CCSR CATEGORY DESCRIPTION IP'
ICD-9 category column: 'CCS CATEGORY'
ICD-9 description column: 'CCS CATEGORY DESCRIPTION'
Combined lookup size: 1558 entries

Before rename, sample 5 feature column names:
  "'1    '" -> Tuberculosis
  "'10   '" -> Immuniz/scrn
  "'100  '" -> Acute MI
  "'101  '" -> Coron athero
  "'102  '" -> Chest pain

After rename, sample 5 feature column names:
  'Tuberculosis (1)'
  'Immuniz/scrn'
  'Acute MI'
  'Coron athero'
  'Chest pain'

Renamed 707 out of 707 diagnosis feature columns

Saved: outputs/ccsr_features.parquet (870.5 KB)
Saved: outputs/ccsr_features_preview.csv (47.5 KB)
Both files confirmed on disk.


In [12]:
os.makedirs("outputs", exist_ok=True)

parquet_path = "outputs/ccsr_features.parquet"
preview_path = "outputs/ccsr_features_preview.csv"

pivot.to_parquet(parquet_path, index=False)
pivot.head(20).to_csv(preview_path, index=False)

print(f"Saved: {parquet_path} ({os.path.getsize(parquet_path) / 1024:.1f} KB)")
print(f"Saved: {preview_path} ({os.path.getsize(preview_path) / 1024:.1f} KB)")
assert os.path.exists(parquet_path), "Parquet missing"
assert os.path.exists(preview_path), "Preview CSV missing"
print("Both files confirmed on disk.")

Saved: outputs/ccsr_features.parquet (870.5 KB)
Saved: outputs/ccsr_features_preview.csv (47.5 KB)
Both files confirmed on disk.
